# Stochastic depression analysis (W-11)

Monte-Carlo uncertainty quantification on depression detection. The method adds Gaussian noise
(σ = supplied vertical-error estimate) to the DEM, runs `fill_depressions` on each noisy
realisation, and aggregates per-cell occurrence probability across N realisations.

**Reference:** Lindsay & Creed (2006). "Distinguishing actual and artefact depressions in digital
elevation data." *Computers & Geosciences* 32(8): 1192–1204.

**Use:** identify which depressions are robust to DEM vertical error vs noise artefacts.

In [ ]:
import numpy as np
from pyramids.dataset import Dataset
from digitalrivers import DEM

# Build a DEM with one clear deep pit + one ambiguous shallow pit.
rows, cols = 20, 20
z = np.full((rows, cols), 100.0, dtype=np.float32)
z[5, 5] = 80.0    # deep pit — 20 m below surroundings
z[15, 15] = 99.0  # shallow pit — only 1 m below surroundings

ds = Dataset.create_from_array(
    z, top_left_corner=(0.0, 0.0), cell_size=30.0,
    epsg=32618, no_data_value=-9999.0,
)
dem = DEM(ds.raster)
print(f"DEM with deep pit at (5, 5)={z[5, 5]}, shallow pit at (15, 15)={z[15, 15]}")

## Low-noise scenario

σ = 0.5 m vertical error (typical for high-quality LiDAR-derived DEMs). The deep pit is far below
any noise realisation; the shallow pit is right at the noise scale.

In [ ]:
prob_low = dem.stochastic_depressions(sigma=0.5, n_runs=50, seed=42).read_array()
print(f"Probability at deep pit    (5, 5):  {prob_low[5, 5]:.3f}")
print(f"Probability at shallow pit (15, 15): {prob_low[15, 15]:.3f}")
print(f"Probability at flat cell   (0, 0):   {prob_low[0, 0]:.3f}")

## High-noise scenario

σ = 5 m vertical error. Now the deep pit's signal-to-noise ratio degrades — it should still
register most of the time, but no longer ~100%.

In [ ]:
prob_high = dem.stochastic_depressions(sigma=5.0, n_runs=50, seed=42).read_array()
print(f"Probability at deep pit    (5, 5):  {prob_high[5, 5]:.3f}")
print(f"Probability at shallow pit (15, 15): {prob_high[15, 15]:.3f}")

## Reproducibility

Same seed → bit-identical output. Useful for reproducible reports.

In [ ]:
p1 = dem.stochastic_depressions(sigma=1.0, n_runs=10, seed=99).read_array()
p2 = dem.stochastic_depressions(sigma=1.0, n_runs=10, seed=99).read_array()
np.testing.assert_array_equal(p1, p2)
print("Same seed produces bit-identical output across calls.")

## Summary

* High-confidence depressions (probability ≈ 1) are robust to DEM noise.
* Marginal cells (probability ~ 0.3–0.7) likely include artefacts.
* Cells with probability ≈ 0 never registered — clearly not pits at this noise scale.

Use the probability raster to threshold a confidence-aware depression mask before downstream
fill / breach operations.